In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


StatementMeta(, ddf2efe2-e322-4d38-af2c-295c3d824001, 3, Finished, Available, Finished, False)

In [2]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

df_bronze = spark.table('bronze_sales')

# --- Quality rules ---
df_clean = (df_bronze
    .filter(F.col('unit_price') > 0)           # drop negative prices
    .filter(F.col('order_id').isNotNull())
    .withColumn('region',
        F.when(F.col('region').isNull(), 'Unknown')
         .otherwise(F.col('region')))
    .withColumn('order_date', F.to_date('order_date'))
    .withColumn('revenue',
        F.round(F.col('quantity') * F.col('unit_price'), 2))
    .withColumn('_loaded_at', F.current_timestamp())
)

print(f'Bronze: {df_bronze.count():,}  Silver: {df_clean.count():,}')
# --- Upsert via MERGE (idempotent reruns) ---
if spark.catalog.tableExists('silver_sales'):
    DeltaTable.forName(spark, 'silver_sales') \
        .alias('t') \
        .merge(df_clean.alias('s'), 't.order_id = s.order_id') \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()
else:
    df_clean.write.format('delta').saveAsTable('silver_sales')

print('Silver MERGE complete')

StatementMeta(, ddf2efe2-e322-4d38-af2c-295c3d824001, 4, Finished, Available, Finished, False)

Bronze: 50,000  Silver: 49,801
Silver MERGE complete
